# ☁️ Cloud URL Health Monitor
### College Cloud Computing Mini-Project

---

**Technology Stack:** Python · Gradio · Requests · Pandas · Google Colab · Google Drive

**Features:**
- ✅ Check website availability in real-time
- ⚡ Measure response times in milliseconds
- 📊 Track HTTP status codes
- 📜 Session monitoring history
- 📥 CSV report export
- ☁️ Google Drive cloud storage

---
> **Instructions:** Run each cell in order from top to bottom. The Gradio interface will launch at the end.


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 1: Install Required Packages                      ║
# ╚══════════════════════════════════════════════════════════╝
#
# This cell installs all dependencies needed to run the app.
# Run time: ~30-60 seconds on first execution.

print('📦 Installing required packages...')
print('This may take up to 60 seconds on first run.\n')

!pip install gradio>=4.0.0 --quiet
!pip install requests>=2.31.0 --quiet
!pip install pandas>=2.0.0 --quiet

print('\n✅ All packages installed successfully!')
print('Proceed to the next cell →')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 2: Import Libraries & Global Configuration        ║
# ╚══════════════════════════════════════════════════════════╝

import requests
import pandas as pd
import gradio as gr
import time
import os
import csv
from datetime import datetime
from urllib.parse import urlparse

# Global session storage for monitoring history
monitoring_history = []

# Column names for the history table and CSV reports
COLUMNS = ['URL', 'Status Code', 'Response Time (ms)', 'Status', 'Checked At']

# Request timeout (seconds)
REQUEST_TIMEOUT = 10

print('✅ Libraries imported successfully!')
print(f'   gradio  version: {gr.__version__}')
print(f'   requests version: {requests.__version__}')
print(f'   pandas  version: {pd.__version__}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 3: Core Application Logic                         ║
# ╚══════════════════════════════════════════════════════════╝
#
# This cell defines:
#   - URL normalization and validation
#   - HTTP health check function
#   - Statistics computation
#   - HTML card builders
#   - Gradio event handlers

# ── URL Utilities ────────────────────────────────────────────────────────────

def normalize_url(url):
    """Adds https:// prefix if no scheme is present."""
    url = url.strip()
    if not url.startswith(('http://', 'https://')):
        url = 'https://' + url
    return url


def validate_url(url):
    """Returns (is_valid, error_message) for a given URL."""
    try:
        parsed = urlparse(url)
        if not parsed.scheme or not parsed.netloc:
            return False, '❌ Invalid URL: Missing scheme or domain.'
        return True, ''
    except Exception as e:
        return False, f'❌ URL Parsing Error: {str(e)}'


# ── Core Health Check ────────────────────────────────────────────────────────

def check_url_health(url):
    """
    Sends an HTTP GET request to the URL and returns a result dictionary
    containing: URL, Status Code, Response Time (ms), Status, Checked At.
    """
    url = normalize_url(url)
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    is_valid, error_msg = validate_url(url)
    if not is_valid:
        return {
            'URL': url, 'Status Code': 'N/A',
            'Response Time (ms)': 'N/A',
            'Status': f'Error – {error_msg}',
            'Checked At': timestamp, 'raw_status': 'error'
        }

    try:
        start = time.time()
        response = requests.get(
            url, timeout=REQUEST_TIMEOUT,
            headers={'User-Agent': 'CloudURLHealthMonitor/1.0 (CollegeProject)'},
            allow_redirects=True
        )
        elapsed_ms = round((time.time() - start) * 1000, 2)
        code = response.status_code

        if 200 <= code < 400:
            status, raw = '🟢 Online', 'online'
        elif 400 <= code < 500:
            status, raw = '🟡 Client Error', 'error'
        elif 500 <= code < 600:
            status, raw = '🔴 Server Error', 'unavailable'
        else:
            status, raw = '⚪ Unknown', 'error'

        return {
            'URL': url, 'Status Code': code,
            'Response Time (ms)': elapsed_ms, 'Status': status,
            'Checked At': timestamp, 'raw_status': raw
        }

    except requests.exceptions.ConnectionError:
        return {
            'URL': url, 'Status Code': 'N/A', 'Response Time (ms)': 'N/A',
            'Status': '🔴 Unavailable – Connection refused',
            'Checked At': timestamp, 'raw_status': 'unavailable'
        }
    except requests.exceptions.Timeout:
        return {
            'URL': url, 'Status Code': 'N/A',
            'Response Time (ms)': f'>{REQUEST_TIMEOUT * 1000}',
            'Status': '⏱️ Unavailable – Timed out',
            'Checked At': timestamp, 'raw_status': 'unavailable'
        }
    except Exception as e:
        return {
            'URL': url, 'Status Code': 'N/A', 'Response Time (ms)': 'N/A',
            'Status': f'❌ Error – {str(e)[:80]}',
            'Checked At': timestamp, 'raw_status': 'error'
        }


# ── Stats & HTML Helpers ─────────────────────────────────────────────────────

def compute_stats(history):
    """Returns aggregate counts and average response time."""
    total = len(history)
    online = sum(1 for r in history if r.get('raw_status') == 'online')
    unavailable = sum(1 for r in history if r.get('raw_status') == 'unavailable')
    numeric_times = [
        float(r['Response Time (ms)'])
        for r in history
        if str(r['Response Time (ms)']).replace('.', '', 1).isdigit()
    ]
    avg = round(sum(numeric_times) / len(numeric_times), 2) if numeric_times else 0
    return {'total': total, 'online': online, 'unavailable': unavailable, 'avg_response': avg}


def build_status_card_html(stats):
    """Returns HTML for the four summary stat cards."""
    return f"""
    <div style="display:flex;gap:16px;flex-wrap:wrap;margin:12px 0;">
        <div style="flex:1;min-width:130px;background:linear-gradient(135deg,#22c55e,#16a34a);
                    border-radius:14px;padding:18px 22px;color:#fff;text-align:center;
                    box-shadow:0 4px 15px rgba(34,197,94,0.35);">
            <div style="font-size:28px;font-weight:800;">{stats['online']}</div>
            <div style="font-size:12px;font-weight:600;opacity:0.9;text-transform:uppercase;">🟢 Online</div>
        </div>
        <div style="flex:1;min-width:130px;background:linear-gradient(135deg,#ef4444,#dc2626);
                    border-radius:14px;padding:18px 22px;color:#fff;text-align:center;
                    box-shadow:0 4px 15px rgba(239,68,68,0.35);">
            <div style="font-size:28px;font-weight:800;">{stats['unavailable']}</div>
            <div style="font-size:12px;font-weight:600;opacity:0.9;text-transform:uppercase;">🔴 Unavailable</div>
        </div>
        <div style="flex:1;min-width:130px;background:linear-gradient(135deg,#8b5cf6,#6d28d9);
                    border-radius:14px;padding:18px 22px;color:#fff;text-align:center;
                    box-shadow:0 4px 15px rgba(139,92,246,0.35);">
            <div style="font-size:28px;font-weight:800;">{stats['total']}</div>
            <div style="font-size:12px;font-weight:600;opacity:0.9;text-transform:uppercase;">📊 Total Checks</div>
        </div>
        <div style="flex:1;min-width:130px;background:linear-gradient(135deg,#0ea5e9,#0284c7);
                    border-radius:14px;padding:18px 22px;color:#fff;text-align:center;
                    box-shadow:0 4px 15px rgba(14,165,233,0.35);">
            <div style="font-size:28px;font-weight:800;">{stats['avg_response']}</div>
            <div style="font-size:12px;font-weight:600;opacity:0.9;text-transform:uppercase;">⚡ Avg ms</div>
        </div>
    </div>
    """


def build_result_card_html(result):
    """Returns a styled HTML card for a single check result."""
    raw = result.get('raw_status', 'error')
    if raw == 'online':
        accent, bg, icon = '#22c55e', 'rgba(34,197,94,0.08)', '✅'
    elif raw == 'unavailable':
        accent, bg, icon = '#ef4444', 'rgba(239,68,68,0.08)', '❌'
    else:
        accent, bg, icon = '#f59e0b', 'rgba(245,158,11,0.08)', '⚠️'

    return f"""
    <div style="border:2px solid {accent};border-radius:16px;padding:20px 24px;
                background:{bg};margin:10px 0;">
        <div style="display:flex;align-items:center;gap:10px;margin-bottom:12px;">
            <span style="font-size:26px;">{icon}</span>
            <div>
                <div style="font-size:17px;font-weight:700;color:#1e293b;word-break:break-all;">
                    {result['URL']}</div>
                <div style="font-size:13px;color:#64748b;">Checked at {result['Checked At']}</div>
            </div>
        </div>
        <div style="display:flex;gap:20px;flex-wrap:wrap;">
            <div style="background:#fff;border-radius:10px;padding:10px 16px;
                        border-left:4px solid {accent};min-width:110px;">
                <div style="font-size:11px;color:#94a3b8;font-weight:600;text-transform:uppercase;">Status</div>
                <div style="font-size:15px;font-weight:700;color:#1e293b;">{result['Status']}</div>
            </div>
            <div style="background:#fff;border-radius:10px;padding:10px 16px;
                        border-left:4px solid #8b5cf6;min-width:110px;">
                <div style="font-size:11px;color:#94a3b8;font-weight:600;text-transform:uppercase;">HTTP Code</div>
                <div style="font-size:15px;font-weight:700;color:#1e293b;">{result['Status Code']}</div>
            </div>
            <div style="background:#fff;border-radius:10px;padding:10px 16px;
                        border-left:4px solid #0ea5e9;min-width:140px;">
                <div style="font-size:11px;color:#94a3b8;font-weight:600;text-transform:uppercase;">Response Time</div>
                <div style="font-size:15px;font-weight:700;color:#1e293b;">{result['Response Time (ms)']} ms</div>
            </div>
        </div>
    </div>
    """


# ── Gradio Event Handlers ────────────────────────────────────────────────────

def handle_check_url(url_input):
    """Handles the Check Website button click event."""
    global monitoring_history

    if not url_input or not url_input.strip():
        return (
            "<div style='color:#ef4444;padding:12px;background:#fef2f2;border-radius:10px;'>"
            "⚠️ Please enter a URL before clicking Check Website.</div>",
            build_status_card_html(compute_stats(monitoring_history)),
            pd.DataFrame(monitoring_history)[COLUMNS] if monitoring_history else pd.DataFrame(columns=COLUMNS),
            '⚠️ No URL provided.'
        )

    raw_urls = [u.strip() for u in url_input.replace('\n', ',').split(',') if u.strip()]
    if len(raw_urls) > 10:
        return (
            "<div style='color:#ef4444;padding:12px;background:#fef2f2;border-radius:10px;'>"
            "⚠️ Maximum 10 URLs can be checked at once.</div>",
            build_status_card_html(compute_stats(monitoring_history)),
            pd.DataFrame(monitoring_history)[COLUMNS] if monitoring_history else pd.DataFrame(columns=COLUMNS),
            '⚠️ Too many URLs.'
        )

    results = []
    for url in raw_urls:
        result = check_url_health(url)
        monitoring_history.append(result)
        results.append(result)

    result_html = ''.join(build_result_card_html(r) for r in results)
    stats = compute_stats(monitoring_history)
    stats_html = build_status_card_html(stats)
    history_df = pd.DataFrame(monitoring_history)[COLUMNS].iloc[::-1].reset_index(drop=True)
    status_msg = f'✅ Checked {len(results)} URL(s) at {datetime.now().strftime("%H:%M:%S")}'
    return result_html, stats_html, history_df, status_msg


def handle_clear_history():
    """Clears all monitoring history."""
    global monitoring_history
    monitoring_history = []
    empty_html = "<div style='color:#64748b;padding:20px;text-align:center;'>No checks performed yet.</div>"
    return (empty_html, build_status_card_html(compute_stats([])),
            pd.DataFrame(columns=COLUMNS), '🗑️ History cleared.')


def handle_generate_report():
    """Generates and returns a downloadable CSV report file."""
    global monitoring_history
    if not monitoring_history:
        return None, '⚠️ No data to export. Please check at least one URL first.'
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f'url_health_report_{timestamp}.csv'
    pd.DataFrame(monitoring_history)[COLUMNS].to_csv(filename, index=False)
    return filename, f"✅ Report saved as '{filename}' – {len(monitoring_history)} records exported."


def handle_save_to_drive(drive_folder):
    """Saves a CSV report to Google Drive (Colab only)."""
    global monitoring_history
    if not monitoring_history:
        return '⚠️ No data to save. Please check at least one URL first.'
    drive_root = '/content/drive/MyDrive'
    if not os.path.exists(drive_root):
        return (
            '⚠️ Google Drive not mounted.\n'
            'Run Cell 4 (Google Drive Setup) first.\n'
            'Code: from google.colab import drive; drive.mount("/content/drive")'
        )
    folder_name = drive_folder.strip() or 'URLHealthMonitor'
    dest_dir = os.path.join(drive_root, folder_name)
    os.makedirs(dest_dir, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    dest_path = os.path.join(dest_dir, f'url_health_report_{timestamp}.csv')
    pd.DataFrame(monitoring_history)[COLUMNS].to_csv(dest_path, index=False)
    return f'☁️ Report saved to Google Drive:\n{dest_path}\n({len(monitoring_history)} records)'


print('✅ Core application logic loaded successfully!')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 4: Google Drive Setup (Optional)                  ║
# ╚══════════════════════════════════════════════════════════╝
#
# Run this cell ONLY if you want to save reports to Google Drive.
# You will be prompted to sign in with your Google account.
# Skip this cell if you only need the CSV download feature.

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('\n✅ Google Drive mounted successfully!')
    print('📁 Your Drive is accessible at: /content/drive/MyDrive')
    print('Reports will be saved to: /content/drive/MyDrive/URLHealthMonitor/')
except ImportError:
    print('ℹ️  Not running in Google Colab – Drive mounting skipped.')
    print('   The "Save to Google Drive" button will show a helpful message.')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 5: Build & Launch Gradio Interface                ║
# ╚══════════════════════════════════════════════════════════╝
#
# This cell builds the complete UI and launches the Gradio app.
# A public URL will appear below – click it to open the dashboard!

CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&display=swap');
body, .gradio-container { font-family: 'Inter', sans-serif !important; background: #f8fafc !important; }
.btn-primary { background: linear-gradient(135deg,#22c55e,#16a34a) !important; color:#fff !important;
               font-weight:700 !important; border-radius:10px !important; border:none !important; }
.btn-secondary { background: linear-gradient(135deg,#8b5cf6,#6d28d9) !important; color:#fff !important;
                 font-weight:600 !important; border-radius:10px !important; border:none !important; }
.btn-danger { background: linear-gradient(135deg,#ef4444,#dc2626) !important; color:#fff !important;
              font-weight:600 !important; border-radius:10px !important; border:none !important; }
"""

HEADER_HTML = """
<div style="background:linear-gradient(135deg,#6d28d9 0%,#4f46e5 40%,#0ea5e9 100%);
            border-radius:20px;padding:32px 40px;margin-bottom:24px;color:#fff;
            text-align:center;box-shadow:0 8px 32px rgba(109,40,217,0.25);">
    <div style="font-size:42px;margin-bottom:8px;">☁️</div>
    <h1 style="margin:0;font-size:30px;font-weight:800;letter-spacing:-0.5px;">Cloud URL Health Monitor</h1>
    <p style="margin:8px 0 0;font-size:15px;opacity:0.85;">Real-time Website Availability &amp; Response Time Monitoring</p>
    <div style="margin-top:14px;display:flex;justify-content:center;gap:14px;flex-wrap:wrap;">
        <span style="background:rgba(255,255,255,0.2);padding:5px 14px;border-radius:20px;
                     font-size:12px;font-weight:600;">⚡ Instant Checks</span>
        <span style="background:rgba(255,255,255,0.2);padding:5px 14px;border-radius:20px;
                     font-size:12px;font-weight:600;">📊 History Tracking</span>
        <span style="background:rgba(255,255,255,0.2);padding:5px 14px;border-radius:20px;
                     font-size:12px;font-weight:600;">📁 CSV Export</span>
        <span style="background:rgba(255,255,255,0.2);padding:5px 14px;border-radius:20px;
                     font-size:12px;font-weight:600;">☁️ Drive Storage</span>
    </div>
</div>
"""

COLUMNS = ['URL', 'Status Code', 'Response Time (ms)', 'Status', 'Checked At']

with gr.Blocks(
    css=CUSTOM_CSS,
    title='Cloud URL Health Monitor',
    theme=gr.themes.Base(
        primary_hue=gr.themes.colors.purple,
        secondary_hue=gr.themes.colors.green,
        neutral_hue=gr.themes.colors.slate,
        font=gr.themes.GoogleFont('Inter'),
    ),
) as demo:

    gr.HTML(HEADER_HTML)

    with gr.Row():
        stats_html = gr.HTML(value=build_status_card_html(compute_stats([])))

    with gr.Group():
        gr.Markdown(
            '### 🔍 Enter URL(s) to Monitor\n'
            '_Enter one URL or multiple URLs separated by commas. Max 10 URLs per check._'
        )
        with gr.Row():
            url_input = gr.Textbox(
                placeholder='e.g. https://google.com, github.com, openai.com',
                label='Website URL(s)', lines=2, max_lines=4, scale=4, elem_id='url-input-box'
            )
            with gr.Column(scale=1, min_width=160):
                check_btn = gr.Button('🔍 Check Website', variant='primary', size='lg', elem_classes=['btn-primary'])
                clear_btn = gr.Button('🗑️ Clear History', variant='stop', size='sm', elem_classes=['btn-danger'])

    status_msg = gr.Textbox(
        label='Status', interactive=False, lines=1,
        value='Ready – Enter a URL above to start monitoring.'
    )

    gr.Markdown('### 📋 Latest Check Result')
    result_html = gr.HTML(
        value="<div style='color:#94a3b8;padding:20px;text-align:center;"
              "border:2px dashed #e2e8f0;border-radius:14px;'>"
              "🔍 Results will appear here after you check a URL.</div>"
    )

    gr.Markdown('### 📜 Monitoring History')
    history_table = gr.Dataframe(
        value=pd.DataFrame(columns=COLUMNS),
        headers=COLUMNS,
        datatype=['str', 'str', 'str', 'str', 'str'],
        interactive=False,
        wrap=True,
        label='All Checks (Most Recent First)',
    )

    with gr.Accordion('📥 Export & Cloud Storage', open=False):
        gr.Markdown('**Generate a CSV report** of all monitoring results, or save to Google Drive.')
        with gr.Row():
            export_btn = gr.Button('📥 Download CSV Report', variant='secondary', elem_classes=['btn-secondary'])
            drive_folder_input = gr.Textbox(label='Google Drive Folder Name', value='URLHealthMonitor', scale=2)
            drive_btn = gr.Button('☁️ Save to Google Drive', variant='secondary', elem_classes=['btn-secondary'])
        report_file = gr.File(label='📁 Download Report')
        report_status = gr.Textbox(label='Export Status', interactive=False, lines=2, value='')

    with gr.Accordion('ℹ️ How to Use This App', open=False):
        gr.Markdown("""
### 📖 Quick Start Guide
| Step | Action |
|------|--------|
| **1** | Enter one or more website URLs in the input box |
| **2** | Click **Check Website** to start monitoring |
| **3** | View the status, HTTP code, and response time |
| **4** | Use **Download CSV Report** to export all results |
| **5** | Use **Save to Google Drive** to store reports in the cloud |

### ☁️ Cloud Computing Concepts Demonstrated
- **Remote HTTP Monitoring** – Cloud-based website health checks
- **Centralized Records** – Session-wide monitoring history
- **Cloud Storage** – Google Drive report persistence
- **Scalability** – Monitor up to 10 URLs simultaneously
        """)

    gr.HTML("""
    <div style="text-align:center;padding:20px;color:#94a3b8;font-size:12px;margin-top:10px;">
        ☁️ Cloud URL Health Monitor &nbsp;|&nbsp; College Cloud Computing Mini-Project &nbsp;|&nbsp;
        Built with Python · Gradio · Requests · Pandas
    </div>
    """)

    outputs = [result_html, stats_html, history_table, status_msg]
    check_btn.click(fn=handle_check_url, inputs=[url_input], outputs=outputs)
    url_input.submit(fn=handle_check_url, inputs=[url_input], outputs=outputs)
    clear_btn.click(fn=handle_clear_history, inputs=[], outputs=outputs)
    export_btn.click(fn=handle_generate_report, inputs=[], outputs=[report_file, report_status])
    drive_btn.click(fn=handle_save_to_drive, inputs=[drive_folder_input], outputs=[report_status])

print('✅ Gradio UI built successfully! Launching now...')
print('━' * 55)
print('  ☁️  Cloud URL Health Monitor')
print('  Click the URL below to open your dashboard!')
print('━' * 55)

# share=True generates a public Gradio link accessible from anywhere
demo.launch(share=True, debug=False, show_error=True)

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 6: Bonus – Command-Line Batch Monitor (Optional)  ║
# ╚══════════════════════════════════════════════════════════╝
#
# Use this cell to quickly check a list of URLs without the UI.
# Useful for demonstrations or quick testing.

DEMO_URLS = [
    'https://www.google.com',
    'https://github.com',
    'https://www.python.org',
    'https://this-url-does-not-exist-xyz.com',   # Expected: Unavailable
    'https://httpstat.us/404',                    # Expected: 404 Client Error
]

print('🔍 Running batch URL health check...\n')
print(f"{'URL':<45} {'Code':<8} {'Time(ms)':<12} {'Status'}")
print('─' * 90)

batch_results = []
for url in DEMO_URLS:
    result = check_url_health(url)
    batch_results.append(result)
    code = str(result['Status Code'])
    t = str(result['Response Time (ms)'])
    st = result['Status']
    display_url = result['URL'][:44]
    print(f"{display_url:<45} {code:<8} {t:<12} {st}")

print('─' * 90)

# Save batch results to CSV
df = pd.DataFrame(batch_results)[COLUMNS]
csv_file = 'batch_health_check.csv'
df.to_csv(csv_file, index=False)

print(f'\n✅ Batch check complete! Results saved to {csv_file}')
print(f'   Total: {len(batch_results)} | Online: {sum(1 for r in batch_results if r["raw_status"]=="online")}'
      f' | Unavailable: {sum(1 for r in batch_results if r["raw_status"]=="unavailable")}')
df